# Classification Workflow

This notebook shows how to run an EpiScope classification task and how to define a new classification task with categories that are not built into the package.


## Setup

A classifier needs four pieces: structured paper records, an index/retriever, a classifier config, and a generator that returns JSON matching the config schema.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import tempfile


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "episcope").exists():
            return candidate
    return None


PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is not None:
    src_path = str(PROJECT_ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

WORK_DIR = Path(tempfile.mkdtemp(prefix="episcope-classification-"))
WORK_DIR


## Build A Small Classification Corpus

The examples use two small papers so the retrieved evidence and final labels are easy to inspect.


In [ ]:
from episcope.schemas import PaperMetadata, StructuredSection

sample_papers = {
    "paper_open_data": {
        "metadata": PaperMetadata(
            title="Trial data sharing and reuse",
            abstract="A randomized trial reused a public patient-level dataset from a hospital registry.",
            keywords=["trial", "registry", "data sharing"],
        ),
        "sections": [
            StructuredSection(
                title="Methods",
                section_type="Methods",
                content=(
                    "The analysis used patient records from the National Hospital Registry. "
                    "The registry stores admission dates, treatment groups, and mortality outcomes."
                ),
            ),
            StructuredSection(
                title="Data availability",
                section_type="Data availability",
                content=(
                    "De-identified trial data and analysis code are available in a public repository. "
                    "The dataset can be reused for non-commercial research after registration."
                ),
            ),
            StructuredSection(
                title="Implications",
                section_type="Discussion",
                content=(
                    "The findings support a hospital guideline for early patient monitoring. "
                    "Implementation would require local workflow changes."
                ),
            ),
        ],
    },
    "paper_methods": {
        "metadata": PaperMetadata(
            title="Reusable screening method for outbreak reports",
            abstract="The paper develops a text-mining method for classifying outbreak reports before manual review.",
            keywords=["method", "screening", "classification"],
        ),
        "sections": [
            StructuredSection(
                title="Method",
                section_type="Methods",
                content=(
                    "We introduce a reusable screening tool and evaluate it on manually annotated abstracts. "
                    "The main contribution is a method for prioritizing review records."
                ),
            ),
            StructuredSection(
                title="Data availability",
                section_type="Data availability",
                content="Annotations are available from the corresponding author upon reasonable request.",
            ),
        ],
    },
}

list(sample_papers)


## Index The Papers

Classification uses the retriever to find evidence for each candidate category. The tiny embedder keeps this notebook offline and deterministic.


In [ ]:
import re
from typing import Iterable

import numpy as np

from episcope.rag.embeddings.base import Embedder


class TinyKeywordEmbedder(Embedder):
    vocabulary = (
        "available",
        "repository",
        "dataset",
        "data",
        "registry",
        "public",
        "request",
        "restricted",
        "confidential",
        "hospital",
        "patient",
        "policy",
        "guideline",
        "method",
        "tool",
        "model",
        "workflow",
        "implementation",
    )

    @property
    def model_name(self) -> str:
        return "tiny-classification-demo"

    @property
    def dim(self) -> int:
        return len(self.vocabulary)

    def embed_text(self, text: str) -> list[float]:
        text = text.lower()
        counts = []
        for term in self.vocabulary:
            pattern = rf"\b{re.escape(term)}s?\b"
            counts.append(float(len(re.findall(pattern, text))))

        vector = np.array(counts, dtype="float32")
        norm = float(np.linalg.norm(vector))
        if norm:
            vector = vector / norm
        return vector.tolist()

    def embed_texts(self, texts: Iterable[str]) -> list[list[float]]:
        return [self.embed_text(text) for text in texts]


embedder = TinyKeywordEmbedder()
embedder.model_name, embedder.dim


In [ ]:
from episcope.db.in_memory_academic_db import InMemoryAcademicDB
from episcope.rag.indexing.chunking import FixedSizeChunker
from episcope.rag.indexing.indexer import Indexer
from episcope.rag.retrieval.candidates import SemanticCandidateRetriever
from episcope.rag.retrieval.retriever import Retriever
from episcope.vectordb.file import FileDB

strategy_name = "classification-demo"
academic_db = InMemoryAcademicDB(backup_file=None)

for paper_id, paper in sample_papers.items():
    academic_db.insert(paper_id, "sections", strategy_name, [section.to_dict() for section in paper["sections"]])
    academic_db.insert(paper_id, "metadata", strategy_name, paper["metadata"].to_dict())
    academic_db.insert(paper_id, "references", strategy_name, [])

vdb = FileDB(str(WORK_DIR / "index"))
indexer = Indexer(
    vdb,
    embedder=embedder,
    chunker=FixedSizeChunker(chunk_size=650, chunk_overlap=75),
)

for paper_id, paper in sample_papers.items():
    indexer.index_paper(paper["sections"], paper["metadata"], paper_id=paper_id)

vdb.save()
semantic_candidates = SemanticCandidateRetriever(vdb, dense_embedder=embedder)
retriever = Retriever(vdb, candidate_retrievers=[semantic_candidates], use_rerank=False)

len(vdb.get_points()), academic_db.list_docs(strategy_name)


## Run A Built-In Classification Task

Built-in configs live in `episcope.workflows.classification`. This example uses `DataAccessibilityClassifierConfig`, which classifies how the paper says its data can be accessed.


In [ ]:
import json
from typing import Any, Callable, Optional, Sequence

from episcope.rag.generation.base import Generator
from episcope.rag.provenance import Provenance


class FixedJSONGenerator(Generator):
    def __init__(self, payload: dict[str, Any], *, model_id: str = "fixed-json-generator"):
        self.payload = payload
        self.model_id = model_id

    def generate(
        self,
        contexts: Sequence[Any],
        *,
        question: Optional[str] = None,
        message_builder: Optional[Callable[..., Any]] = None,
        **kwargs: Any,
    ) -> Provenance:
        return Provenance(answer=json.dumps(self.payload), evidences=[])


In [ ]:
from episcope.workflows import PaperClassifier
from episcope.workflows.classification import DataAccessibilityClassifierConfig

availability_config = DataAccessibilityClassifierConfig(top_k=4)
availability_payload = {
    "classification": ["A"],
    "primary_label": "A",
    "reasoning": "The paper states that de-identified trial data and code are available in a public repository.",
    "confidence": 0.92,
    "class_probabilities": {
        "A": 0.92,
        "B": 0.03,
        "C": 0.02,
        "D": 0.01,
        "E": 0.01,
        "F": 0.01,
    },
}

availability_classifier = PaperClassifier(
    retriever=retriever,
    generator=FixedJSONGenerator(availability_payload, model_id="demo-data-availability"),
    strategy_name=strategy_name,
    config=availability_config,
    academic_db=academic_db,
)

availability_result = availability_classifier.run_detailed("paper_open_data")
[label.value for label in availability_result.decision.result.classification]


## Inspect The Classification Trace

Use `run_detailed` while developing a task. It exposes retrieved evidence, prompt messages, raw generator output, parsed labels, and training-oriented artifacts.


In [ ]:
print("Classification:", [label.value for label in availability_result.decision.result.classification])
print("Confidence:", availability_result.decision.result.confidence)
print("Reasoning:", availability_result.decision.result.evidence["reasoning"])

print("\nTop evidence")
for chunk in availability_result.decision.top_evidence:
    print(f"- category={chunk.artifacts.get('category')} | {chunk.paper_id} | {chunk.section_title}")
    print(" ", chunk.text[:180])

print("\nRaw generator output")
print(availability_result.trace.raw_llm_response)


## Define A New Classification Task

A custom classifier is a `BaseClassifierConfig` plus a Pydantic output schema. The example below creates a review-routing task with three categories: action-oriented papers, methods/tool papers, and manual review.


In [ ]:

from typing import List, Literal

from pydantic import Field, model_validator

from episcope.workflows.classification import BaseClassifierConfig
from episcope.workflows.classification.schemas import BaseClassificationSchema

ReviewRouteCode = Literal["A", "B", "C"]


class ReviewRouteOutput(BaseClassificationSchema):
    classification: List[ReviewRouteCode] = Field(
        ...,
        min_length=1,
        description="A list with one review-route code: A, B, or C.",
    )
    primary_label: Optional[ReviewRouteCode] = Field(
        default=None,
        description="The dominant review-route code.",
    )

    @model_validator(mode="after")
    def normalize(self) -> "ReviewRouteOutput":
        canonical = [code for code in ["A", "B", "C"] if code in self.classification]
        if not canonical:
            canonical = ["C"]
        self.classification = [canonical[0]]
        self.primary_label = self.primary_label if self.primary_label in self.classification else self.classification[0]
        return self


ReviewRouteOutput.model_rebuild()


review_route_labels = {
    "A": "Action or policy relevant",
    "B": "Method or tool",
    "C": "Manual review",
}
review_route_definitions = {
    "A": "Directly supports action, policy, guidance, or implementation.",
    "B": "Develops or evaluates a method, tool, model, or workflow.",
    "C": "Needs manual review because the route is ambiguous or unsupported.",
}

custom_config = BaseClassifierConfig(
    top_k=4,
    template_paragraphs={
        "action_or_policy": [
            "The findings support a policy, guideline, implementation plan, or operational decision.",
            "The paper provides evidence that can be used for public health action or clinical workflow changes.",
        ],
        "method_or_tool": [
            "The main contribution is a method, tool, model, screening system, or reusable workflow.",
            "The paper develops or evaluates a methodological approach rather than making a direct policy recommendation.",
        ],
        "manual_review": [
            "The paper is ambiguous, background-only, or lacks enough information for automatic routing.",
        ],
    },
    classification_mapping={
        "A": "action_or_policy",
        "B": "method_or_tool",
        "C": "manual_review",
    },
    category_labels=review_route_labels,
    system_prompt=(
        "You route epidemiology papers for review. Choose the route that best reflects "
        "the paper's main contribution, using only the provided text."
    ),
    user_prompt_template=r'''
Classify the paper into exactly one review route.

Categories:
{categories}

Definitions:
{definitions}

Paper content:
Title: {title}
Abstract: {abstract}
Keywords: {keywords}

Relevant extracts:
{chunks_info}

Instructions:
1. Choose A when the paper directly supports action, policy, guidance, or implementation.
2. Choose B when the main contribution is a method, tool, model, or reusable workflow.
3. Choose C when the evidence is insufficient or the paper needs manual review.
4. Return only a JSON object matching this schema:
{schema}
''',
    extra_output_fields={
        "definitions": "\n".join(
            f"{code}: {definition}" for code, definition in review_route_definitions.items()
        )
    },
    output_schema=ReviewRouteOutput,
    default_classification=["manual_review"],
)

custom_config.category_labels


## Run The Custom Classifier

The workflow is the same as the built-in task. Only the config, schema, and expected generator JSON changed.


In [ ]:
custom_payload = {
    "classification": ["B"],
    "primary_label": "B",
    "reasoning": "The paper's main contribution is a reusable screening method for prioritizing review records.",
    "confidence": 0.88,
    "class_probabilities": {"A": 0.08, "B": 0.88, "C": 0.04},
}

custom_classifier = PaperClassifier(
    retriever=retriever,
    generator=FixedJSONGenerator(custom_payload, model_id="demo-review-route"),
    strategy_name=strategy_name,
    config=custom_config,
    academic_db=academic_db,
)

custom_result = custom_classifier.run_detailed("paper_methods")
custom_result.decision.result.classification


In [ ]:
print("Classification:", custom_result.decision.result.classification)
print("Reasoning:", custom_result.decision.result.evidence["reasoning"])
print("Prompt messages:", len(custom_result.trace.prompt_messages))
print("Raw JSON:", custom_result.trace.raw_llm_response)

print("\nEvidence categories retrieved for the task")
for chunk in custom_result.decision.top_evidence:
    print(f"- {chunk.artifacts.get('category')} | {chunk.section_title}: {chunk.text[:150]}")


## Checklist For New Categories

To add your own classification task, define the label set first, then keep the config components aligned:

1. `category_labels`: the codes and human-readable names shown in the prompt.
2. `template_paragraphs`: retrieval queries or prototype snippets for each category.
3. `output_schema`: the JSON shape the generator must return.
4. `classification_mapping`: how schema codes map to final labels.
5. `default_classification`: the fallback label when parsing fails.
6. `system_prompt` and `user_prompt_template`: the decision rules, definitions, and schema instructions.

For real model calls, replace `FixedJSONGenerator` with `LLMGenerator` or another `Generator` implementation that returns valid JSON for the selected schema.
